# Exploring the Dataset: Building the VPN Logs

**Goal:** Understand how we go from a raw LOG configuration file to a relational database table.

This notebook walks through:
1. Loading `vpn/logs/openvpn.log` (contains access vpn logs for vpn_events table)
2. Examining what fields each access log
3. Transforming the data into a Pandas DataFrame

---

**Dataset:** AIT Log Data Set V2.0 — russellmitchell testbed  
**Source:** https://zenodo.org/records/5789064

## 0. Configuration

Set the path to the `russellmitchell/` dataset folder below.  

**Default:** Assumes `russellmitchell/` is at the same level as the repo:
```
data-201-group-project/
├── data-201-security-log-analysis/   <-- this repo
│   └── notebooks/                    <-- this notebook is here
└── russellmitchell/                  <-- dataset is here
```

If your dataset is somewhere else, just change `DATASET_ROOT` below.

In [17]:
import re
from pathlib import Path

import pandas as pd

# --- CHANGE THIS if your dataset is in a different location ---
DATASET_ROOT = Path("..") / ".." / "russellmitchell"

# Verify the path exists
if not DATASET_ROOT.exists():
    print(f"ERROR: Dataset not found at: {DATASET_ROOT.resolve()}")
    print("")
    print("Expected directory structure:")
    print("  <workspace>/data-201-security-log-analysis/notebooks/  <-- you are here")
    print("  <workspace>/russellmitchell/                          <-- dataset should be here")
    print("")
    print("Fix: Update DATASET_ROOT above to point to your russellmitchell/ folder.")
else:
    print(f"Dataset found at: {DATASET_ROOT.resolve()}")

Dataset found at: /Users/rrosevearehunt/PythonProjects/DATA201/GroupProject/russellmitchell


## 1. Load the Raw log File and create Data Frame

The file `gather/vpn/logs/openvpn.log` 

In [18]:
openvpn_log = DATASET_ROOT / "gather" / "vpn" / "logs" / "openvpn.log"

def parse_openvpn_logs(file_path):
    # Regex Pattern Breakdown:
    # Group 1: Date (e.g., '2022-01-21')
    # Group 2: Time (e.g., '00:09:11')
    # Group 3: Username (Optional, e.g., 'jhall'). The '(?:([^/\s]+)/)?' handles the optional "username/" prefix
    # Group 4: Source IP (e.g., '192.168.230.165')
    # Group 5: Source Port (e.g., '46011')
    # Group 6: Message (The rest of the string)
    log_pattern = re.compile(
        r'^(\d{4}-\d{2}-\d{2})\s+(\d{2}:\d{2}:\d{2})\s+(?:([^/\s]+)/)?(\d{1,3}(?:\.\d{1,3}){3}):(\d+)\s+(.*)$'
    )

    parsed_data = []

    try:
        with open(file_path) as file:
            for line in file:
                line = line.strip()

                # Skip empty lines
                if not line:
                    continue

                # Ignore metadata tags if they exist in your raw file (like )
                if line.startswith('['):
                    # For handling broken lines from copy-pasting, you might need to clean this up,
                    # but standard OpenVPN logs won't have these tags.
                    continue

                match = log_pattern.search(line)
                if match:
                    date, time, username, ip, port, message = match.groups()

                    # Combine Date and Time into a single Pandas DateTime object
                    event_timestamp = pd.to_datetime(f"{date} {time}")

                    parsed_data.append({
                        'event_timestamp': event_timestamp,
                        'username': username if username else None, # Nullable for pre-auth packets
                        'source_ip': ip,
                        'source_port': int(port),
                        'message': message.strip()
                    })

    except FileNotFoundError:
        print(f"Error: The file {file_path} was not found.")
        return None

    # Convert to Pandas DataFrame
    df = pd.DataFrame(parsed_data)

    # Cast the source_port to Pandas' safe integer type (allows NaN/NULLs if needed later)
    df['source_port'] = df['source_port'].astype('Int64')

    return df

# Execute the parser
openvpn_df = parse_openvpn_logs(openvpn_log)

# Print the first few rows to verify the structure
print(openvpn_df.head(10))

# Export exactly as it needs to go into PostgreSQL
openvpn_df.to_csv('openvpn_logs_structured.csv', index=False)

      event_timestamp username        source_ip  source_port  \
0 2022-01-21 00:09:11    jhall  192.168.230.165        46011   
1 2022-01-21 00:09:11    jhall  192.168.230.165        46011   
2 2022-01-21 00:09:11    jhall  192.168.230.165        46011   
3 2022-01-21 00:09:11    jhall  192.168.230.165        46011   
4 2022-01-21 00:09:11    jhall  192.168.230.165        46011   
5 2022-01-21 00:09:11    jhall  192.168.230.165        46011   
6 2022-01-21 00:09:11    jhall  192.168.230.165        46011   
7 2022-01-21 00:09:11    jhall  192.168.230.165        46011   
8 2022-01-21 00:09:11    jhall  192.168.230.165        46011   
9 2022-01-21 00:09:11    jhall  192.168.230.165        46011   

                                                       message  
0      TLS: soft reset sec=3308/3308 bytes=45748/-1 pkts=649/0  
1  VERIFY OK: depth=1, C=AT, ST=Vienna, L=Vienna, O=Some Or...  
2                                                 VERIFY KU OK  
3                    Validating cer

## 2. Invesigate jhall

Since we know jhall is compromised from the authorization logs, lets look at what jhall did through the vpn

In [19]:
# Filter for only 'jhall'
jhall_logs = openvpn_df[openvpn_df['username'] == 'jhall']

# See all unique IPs jhall connected from and how many logs each generated
print(jhall_logs['source_ip'].value_counts())

# See the timeline of jhall's connections
jhall_connections = jhall_logs[jhall_logs['message'].str.contains('Peer Connection Initiated', na=False)]
print(jhall_connections[['event_timestamp', 'source_ip']])

source_ip
192.168.230.165    2170
192.168.230.122      32
Name: count, dtype: int64
Empty DataFrame
Columns: [event_timestamp, source_ip]
Index: []


## 3. Map External IPs to Internal VPN IPs

When a user connects to OpenVPN, they are assigned an internal IP address (e.g., 10.9.0.x). Extracting this mapping is critical for correlating VPN logs with internal server logs (like firewall logs or web server access logs).

Exploration: Extract the assigned internal IP from the message column and map it to the user.


In [20]:
# Filter for lines where the VPN pool assigns an IP
ip_assignments = openvpn_df[openvpn_df['message'].str.contains('MULTI_sva: pool returned IPv4=', na=False)].copy()

# Extract the internal IP using a quick regex
ip_assignments['internal_vpn_ip'] = ip_assignments['message'].str.extract(r'IPv4=([\d\.]+)')

# Display the Mapping: Time -> User -> External IP -> Internal IP
print(ip_assignments[['event_timestamp', 'username', 'source_ip', 'internal_vpn_ip']])

         event_timestamp username        source_ip internal_vpn_ip
166  2022-01-21 06:30:01   twhite   192.168.230.95        10.9.0.6
280  2022-01-21 08:49:04   ahayes  192.168.231.127       10.9.0.10
329  2022-01-21 09:40:01    jhall  192.168.230.165       10.9.0.14
380  2022-01-21 09:55:46   ahayes  192.168.231.127        10.9.0.6
431  2022-01-21 10:48:30   twhite   192.168.230.95       10.9.0.10
...                  ...      ...              ...             ...
5214 2022-01-24 16:05:11   twhite   192.168.230.95       10.9.0.10
5266 2022-01-24 16:59:53   ahayes  192.168.231.127       10.9.0.14
5317 2022-01-24 17:13:07   ahayes  192.168.231.127       10.9.0.14
5347 2022-01-24 17:45:03   ahayes  192.168.231.127       10.9.0.14
5400 2022-01-24 18:13:02   ahayes  192.168.231.127       10.9.0.10

[66 rows x 4 columns]


## 4. Credential Sharing or Account Takeover

A common indicator of compromise is "Impossible Travel" or a single user account connecting from multiple different external IPs in a short timeframe.

Exploration: Count how many unique IP addresses are associated with each username.


In [22]:
# Group by username and count unique source IPs
user_ip_counts = openvpn_df.groupby('username')['source_ip'].nunique().sort_values(ascending=False)

print("Number of unique IPs per user:")
print(user_ip_counts)

# If a user has multiple IPs, list them out:
suspicious_users = user_ip_counts[user_ip_counts > 1].index
print("\nDetails for users with multiple IPs:")
print(openvpn_df[openvpn_df['username'].isin(suspicious_users)].groupby(['username', 'source_ip']).size())

Number of unique IPs per user:
username
jhall     2
ahayes    1
twhite    1
Name: source_ip, dtype: int64

Details for users with multiple IPs:
username  source_ip      
jhall     192.168.230.122      32
          192.168.230.165    2170
dtype: int64


## 5. Identify Connection Errors and Scanners

Attackers often scan for VPN endpoints or attempt to connect without the proper TLS certificates. This generates specific error messages in the logs.

Exploration: Filter for connection drops, TLS errors, and timeouts.

In [23]:
# Look for common error keywords in the message column
error_keywords = 'Error|failed|timeout|reset'
errors_df = openvpn_df[openvpn_df['message'].str.contains(error_keywords, case=False, na=False)]

# See the most common types of errors
print("Top Error Messages:")
print(errors_df['message'].value_counts().head(5))

# See which IPs are generating the most errors (potential scanners/attackers)
print("\nIPs Generating the Most Errors:")
print(errors_df['source_ip'].value_counts().head(5))

Top Error Messages:
message
[twhite] Inactivity timeout (--ping-restart), restarting                                              32
[ahayes] Inactivity timeout (--ping-restart), restarting                                              31
TLS: soft reset sec=3261/3261 bytes=45260/-1 pkts=642/0                                                4
TLS Error: TLS key negotiation failed to occur within 60 seconds (check your network connectivity)     3
TLS Error: TLS handshake failed                                                                        3
Name: count, dtype: int64

IPs Generating the Most Errors:
source_ip
192.168.230.165    110
192.168.230.95      72
192.168.231.127     59
192.168.230.122      2
Name: count, dtype: int64


## 5. Calculate Activity Volume Over Time (Visual Exploration)

Understanding when people normally connect to the VPN establishes a baseline. Connections at 3:00 AM on a Sunday might warrant investigation.

Exploration: Group the logs by hour to see the traffic distribution.

In [26]:
# Create the hourly traffic series without permanently altering openvpn_df
if 'event_timestamp' in openvpn_df.columns:
    # Temporarily set the index just for this calculation
    hourly_traffic = openvpn_df.set_index('event_timestamp').resample('h').size()
else:
    # If it's already the index (from a previous run), just resample directly
    hourly_traffic = openvpn_df.resample('h').size()

print("Hourly Traffic Volume:")
print(hourly_traffic)

# Optional: Plot the results (if you have matplotlib installed)
# hourly_traffic.plot(title="VPN Log Volume Over Time", figsize=(10, 5))

Hourly Traffic Volume:
event_timestamp
2022-01-21 00:00:00    21
2022-01-21 01:00:00    42
2022-01-21 02:00:00    21
2022-01-21 03:00:00    21
2022-01-21 04:00:00    21
                       ..
2022-01-24 19:00:00    21
2022-01-24 20:00:00    21
2022-01-24 21:00:00    21
2022-01-24 22:00:00    21
2022-01-24 23:00:00    21
Freq: h, Length: 96, dtype: int64


## 6. Display full table


In [27]:
# Display the full table
pd.set_option("display.max_colwidth", 60)
pd.set_option("display.max_columns", None)
openvpn_df

,username,source_ip,source_port,message
event_timestamp,,,,
2022-01-21 00:09:11,jhall,192.168.230.165,46011,TLS: soft reset sec=3308/3308 bytes=45748/-1 pkts=649/0
2022-01-21 00:09:11,jhall,192.168.230.165,46011,"VERIFY OK: depth=1, C=AT, ST=Vienna, L=Vienna, O=Some Or..."
2022-01-21 00:09:11,jhall,192.168.230.165,46011,VERIFY KU OK
2022-01-21 00:09:11,jhall,192.168.230.165,46011,Validating certificate extended key usage
2022-01-21 00:09:11,jhall,192.168.230.165,46011,++ Certificate has EKU (str) TLS Web Client Authenticati...
...,...,...,...,...
2022-01-24 23:12:27,jhall,192.168.230.165,59384,Outgoing Data Channel: Cipher 'AES-256-CBC' initialized ...
2022-01-24 23:12:27,jhall,192.168.230.165,59384,Outgoing Data Channel: Using 160 bit message hash 'SHA1'...
2022-01-24 23:12:27,jhall,192.168.230.165,59384,Incoming Data Channel: Cipher 'AES-256-CBC' initialized ...


In [30]:
# Query the DataFrame with SQL (pandasql runs SQL on the in-memory table)

from pandasql import sqldf

# 1. Add the "FROM openvpn_df" to the SQL string
query = """
    SELECT event_timestamp, username, source_ip, source_port, message
    FROM openvpn_df
    ORDER BY event_timestamp ASC;
"""

# 2. Pass globals() so pandasql can locate the 'openvpn_df' variable in memory
sorted_logs_df = sqldf(query, globals())

# Print the first few rows to verify
print(sorted_logs_df.head())

              event_timestamp username        source_ip  source_port  \
0  2022-01-21 00:09:11.000000    jhall  192.168.230.165        46011   
1  2022-01-21 00:09:11.000000    jhall  192.168.230.165        46011   
2  2022-01-21 00:09:11.000000    jhall  192.168.230.165        46011   
3  2022-01-21 00:09:11.000000    jhall  192.168.230.165        46011   
4  2022-01-21 00:09:11.000000    jhall  192.168.230.165        46011   

                                                       message  
0      TLS: soft reset sec=3308/3308 bytes=45748/-1 pkts=649/0  
1  VERIFY OK: depth=1, C=AT, ST=Vienna, L=Vienna, O=Some Or...  
2                                                 VERIFY KU OK  
3                    Validating certificate extended key usage  
4  ++ Certificate has EKU (str) TLS Web Client Authenticati...  


## 7. Mapping to the Database Schema

Here's how this log data maps to our planned **`vpn_log`** table in PostgreSQL:

| LOG field | DB Column | SQL Type | Notes |
|-----------|-----------|----------|-------|
| *(auto-generated)* | `vpn_log_id` | `SERIAL PRIMARY KEY` | Auto-incrementing ID |
| `event_timestamp` | `event_timestamp` | `TIMESTAMP NOT NULL` | timestamp of the event|
| `username` | `username` | `VARCHAR(100)` | host |
| `source_ip` | `source_ip` | `INET` | Derived: CRON, sudo, sshd, systemd=logind, su, systemd |
| `source_port` | `source_port` | `INTEGER` | Log message |
| `message` | `message` | `TEXT` | vpn message |
| *(auto-generated)* | `created_at` | `TIMESTAMP DEFAULT CURRENT_TIMESTAMP` | When this row was inserted |

### The SQL `CREATE TABLE` statement:

```sql
CREATE TABLE auth_logs (
    vpn_log_id SERIAL PRIMARY KEY,
    event_timestamp TIMESTAMP NOT NULL,
    username VARCHAR(100),
    source_ip INET,
    source_port INT,
    message TEXT,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
);
```
